This is a notebook for training using CatBoostRegressor (CBR).

In [1]:
import pandas as pd
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import root_mean_squared_error
from catboost import CatBoostRegressor

In [2]:
# Load the data
# train_df = pd.read_csv('./inputs/train.csv').set_index("id")
%store -r train_df
train_df = train_df
target = train_df["Listening_Time_minutes"]
train_df.drop("Listening_Time_minutes", axis=1, inplace=True)
# train_df.drop("Episode_Title", axis=1, inplace=True)
train_df

,Podcast_Name,Episode_Title,Episode_Length_minutes,Genre,Host_Popularity_percentage,Publication_Day,Publication_Time,Guest_Popularity_percentage,Number_of_Ads,Episode_Sentiment
id,,,,,,,,,,
0,Mystery Matters,Episode 98,46.5033,True Crime,74.81,Thursday,Night,56.8075,0.0,Positive
1,Joke Junction,Episode 26,119.8000,Comedy,66.95,Saturday,Afternoon,75.9500,2.0,Negative
2,Study Sessions,Episode 16,73.9000,Education,69.97,Tuesday,Evening,8.9700,0.0,Negative
3,Digital Digest,Episode 45,67.1700,Technology,57.22,Monday,Morning,78.7000,2.0,Positive
4,Mind & Body,Episode 86,110.5100,Health,80.07,Monday,Afternoon,58.6800,3.0,Neutral
...,...,...,...,...,...,...,...,...,...,...
749995,Learning Lab,Episode 25,75.6600,Education,69.36,Saturday,Morning,55.0653,0.0,Negative
749996,Business Briefs,Episode 21,75.7500,Business,35.21,Saturday,Night,49.5326,2.0,Neutral
749997,Lifestyle Lounge,Episode 51,30.9800,Lifestyle,78.58,Thursday,Morning,84.8900,0.0,Negative


In [3]:
# Encode the categorical columns
%store -r categories
# categories = categories[:1] + categories[2:]
display(categories)
encoder = LabelEncoder()
for column in categories:
    train_df[column] = encoder.fit_transform(train_df[column])

train_df[categories]

Index(['Podcast_Name', 'Episode_Title', 'Genre', 'Publication_Day',
       'Publication_Time', 'Episode_Sentiment'],
      dtype='object')

,Podcast_Name,Episode_Title,Genre,Publication_Day,Publication_Time,Episode_Sentiment
id,,,,,,
0,34,98,9,4,3,2
1,24,19,1,2,0,0
2,40,8,2,5,1,0
3,10,40,8,1,2,2
4,31,85,3,1,0,1
...,...,...,...,...,...,...
749995,26,18,2,2,2,0
749996,2,14,0,2,3,1
749997,28,47,4,4,2,0


In [4]:
# Scale the data
scaler = StandardScaler()
train_df_scaled = scaler.fit_transform(train_df)
train_df_scaled

array([[ 0.74158935,  1.69278629, -0.5419192 , ...,  0.18069978,
        -1.17176445,  1.22882316],
       [ 0.03425408, -1.10995773,  1.70156706, ...,  0.92637313,
         0.56565891, -1.22384261],
       [ 1.16599051, -1.50021322,  0.2966468 , ..., -1.68275323,
        -1.17176445, -1.22384261],
       ...,
       [ 0.31718819, -0.1165801 , -1.01706078, ...,  1.27462022,
        -1.17176445, -1.22384261],
       [ 1.23672404, -0.29396896,  1.37038542, ...,  1.60105317,
        -1.17176445, -1.22384261],
       [ 1.02452346,  1.72826406, -1.22764578, ..., -0.60178493,
        -1.17176445,  0.00249027]])

In [5]:
X_train, X_test, y_train, y_test = train_test_split(train_df_scaled, target, test_size=0.2, random_state=42)

In [6]:
# Create the model
model = CatBoostRegressor()
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
score = root_mean_squared_error(y_test, y_pred)
print(score)

Learning rate set to 0.112494
0:	learn: 24.6418725	total: 191ms	remaining: 3m 10s
1:	learn: 22.4592559	total: 238ms	remaining: 1m 58s
2:	learn: 20.5603507	total: 283ms	remaining: 1m 33s
3:	learn: 18.9172397	total: 321ms	remaining: 1m 19s
4:	learn: 17.4953844	total: 361ms	remaining: 1m 11s
5:	learn: 16.2856769	total: 406ms	remaining: 1m 7s
6:	learn: 15.2574615	total: 464ms	remaining: 1m 5s
7:	learn: 14.3793461	total: 506ms	remaining: 1m 2s
8:	learn: 13.6387467	total: 548ms	remaining: 1m
9:	learn: 13.0112737	total: 588ms	remaining: 58.2s
10:	learn: 12.4880019	total: 633ms	remaining: 56.9s
11:	learn: 12.0521945	total: 688ms	remaining: 56.6s
12:	learn: 11.6969587	total: 735ms	remaining: 55.8s
13:	learn: 11.4056528	total: 780ms	remaining: 54.9s
14:	learn: 11.1655526	total: 819ms	remaining: 53.8s
15:	learn: 10.9684523	total: 861ms	remaining: 53s
16:	learn: 10.8100753	total: 945ms	remaining: 54.7s
17:	learn: 10.6761824	total: 986ms	remaining: 53.8s
18:	learn: 10.5696873	total: 1.03s	remaining

In [13]:
# Trying to find the best parameters for the CatBoost Model using GridSearchCV
parameter = {
    "random_state": [21, 34, 42, 50, 1],
}
base_model = CatBoostRegressor(n_estimators=1000, learning_rate=0.11249399930238724, max_depth=6)
grid = GridSearchCV(estimator=base_model, param_grid=parameter, scoring='neg_root_mean_squared_error', n_jobs=-1,
                    verbose=3)
grid.fit(X_train, y_train)
predictions = grid.predict(X_test)
score = root_mean_squared_error(y_test, y_pred)
print(score)

Fitting 5 folds for each of 5 candidates, totalling 25 fits
0:	learn: 24.6391500	total: 63.2ms	remaining: 1m 3s
1:	learn: 22.4617009	total: 121ms	remaining: 1m
2:	learn: 20.5646176	total: 179ms	remaining: 59.5s
3:	learn: 18.9224517	total: 248ms	remaining: 1m 1s
4:	learn: 17.5032884	total: 325ms	remaining: 1m 4s
5:	learn: 16.2866912	total: 384ms	remaining: 1m 3s
6:	learn: 15.2399863	total: 451ms	remaining: 1m 3s
7:	learn: 14.3635708	total: 518ms	remaining: 1m 4s
8:	learn: 13.6154723	total: 590ms	remaining: 1m 4s
9:	learn: 12.9952774	total: 656ms	remaining: 1m 4s
10:	learn: 12.4727782	total: 719ms	remaining: 1m 4s
11:	learn: 12.0479063	total: 783ms	remaining: 1m 4s
12:	learn: 11.6978792	total: 852ms	remaining: 1m 4s
13:	learn: 11.4063220	total: 910ms	remaining: 1m 4s
14:	learn: 11.1642215	total: 969ms	remaining: 1m 3s
15:	learn: 10.9694541	total: 1.03s	remaining: 1m 3s
16:	learn: 10.8069913	total: 1.09s	remaining: 1m 3s
17:	learn: 10.6765070	total: 1.16s	remaining: 1m 3s
18:	learn: 10.56

In [14]:
grid.best_params_

{'random_state': 34}

In [15]:
grid.best_score_

-9.932770537104668

In [8]:
model.get_all_params()

{'nan_mode': 'Min',
 'eval_metric': 'RMSE',
 'iterations': 1000,
 'sampling_frequency': 'PerTree',
 'leaf_estimation_method': 'Newton',
 'random_score_type': 'NormalWithModelSizeDecrease',
 'grow_policy': 'SymmetricTree',
 'penalties_coefficient': 1,
 'boosting_type': 'Plain',
 'model_shrink_mode': 'Constant',
 'feature_border_type': 'GreedyLogSum',
 'bayesian_matrix_reg': 0.10000000149011612,
 'eval_fraction': 0,
 'force_unit_auto_pair_weights': False,
 'l2_leaf_reg': 3,
 'random_strength': 1,
 'rsm': 1,
 'boost_from_average': True,
 'model_size_reg': 0.5,
 'pool_metainfo_options': {'tags': {}},
 'subsample': 0.800000011920929,
 'use_best_model': False,
 'random_seed': 0,
 'depth': 6,
 'posterior_sampling': False,
 'border_count': 254,
 'classes_count': 0,
 'auto_class_weights': 'None',
 'sparse_features_conflict_fraction': 0,
 'leaf_estimation_backtracking': 'AnyImprovement',
 'best_model_min_trees': 1,
 'model_shrink_rate': 0,
 'min_data_in_leaf': 1,
 'loss_function': 'RMSE',
 'lear

In [17]:
display(model.score(X_test, y_test))
display(model.score(X_train, y_train))

0.8665673340117628

0.8702034823360925